In [1]:
# ============================================================
# Phi-3 delimiter causality experiment
# Clean:  "## Question:"
# Corrupt: "Question:"
#
# Saves:
#   - per-sample generation results CSVs
#   - tokenization CSVs
#   - activation patching CSVs
#   - DLA CSVs
#   - attention CSVs
#   - summary JSON + Markdown report
#   - plots (accuracy, logprob, patching, DLA, attention, heatmaps)
# ============================================================

import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import json
import math
import random
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    from peft import PeftModel
    HAS_PEFT = True
except Exception:
    HAS_PEFT = False

# ============================================================
# CONFIG
# ============================================================

BASE_MODEL = "microsoft/phi-3-mini-4k-instruct"
SFT_PATH = "/kaggle/input/datasets/adithyaled24b039/phi3-sft-folderr/phi3_sft_lora"

OUT_DIR = "/kaggle/working/phi3_delimiter_causality"
TASK_DIRS = {
    "gsm8k": os.path.join(OUT_DIR, "gsm8k"),
    "strategyqa": os.path.join(OUT_DIR, "strategyqa"),
    "mmlu": os.path.join(OUT_DIR, "mmlu"),
}
PLOTS_DIR = os.path.join(OUT_DIR, "plots")
REPORTS_DIR = os.path.join(OUT_DIR, "reports")
CACHE_DIR = os.path.join(OUT_DIR, "eval_cache")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

SEED = 42
EVAL_SEED = 42

# Keep this small enough for a Kaggle T4 run
N_GSM8K = 8
N_STRATEGYQA = 8
N_MMLU_PER_SUBJECT = 4

MMLU_SUBJECTS = [
    "abstract_algebra",
    "college_mathematics",
    "logical_fallacies",
]

# Prompt / analysis controls
USE_FEWSHOT = False   # keep False so "## Question:" sits at the prompt start
MAX_NEW_GSM8K = 128
MAX_NEW_STRATEGYQA = 32
MAX_NEW_MMLU = 64

PATCH_WINDOW = 6                  # patch first N prompt tokens (delimiter region)
POSITION_SWEEP_LAYERS = 5        # number of layers to sweep in the positional patch heatmap
ATTN_PREFIX_WINDOW = 6           # attention mass to the first N prompt tokens

# If True, tries to load and merge the SFT LoRA adapter.
USE_SFT_ADAPTER = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
for d in TASK_DIRS.values():
    os.makedirs(d, exist_ok=True)

TOKENIZER = None
MODEL = None
BLOCKS = None
FINAL_NORM = None
LM_HEAD = None

# ============================================================
# PROMPT TEMPLATES
# ============================================================

SYSTEM_STYLE_NOTE = (
    "Solve carefully. Think step by step if helpful, and put the final answer "
    "inside <answer></answer>."
)

def build_prompt(task: str, question: str, clean: bool = True, choices=None) -> str:
    """
    Plain-text prompt by design:
    - keeps '## Question:' or 'Question:' at the very start
    - makes the delimiter location easy to study causally
    """
    header = "## Question:" if clean else "Question:"

    if task == "gsm8k":
        prompt = (
            f"{header} {question}\n"
            f"{SYSTEM_STYLE_NOTE}\n"
            "Put ONLY the final numeric answer inside <answer></answer>."
        )
        return prompt

    if task == "strategyqa":
        prompt = (
            f"{header} {question}\n"
            f"{SYSTEM_STYLE_NOTE}\n"
            "Answer with exactly yes or no inside <answer></answer>."
        )
        return prompt

    if task == "mmlu":
        assert choices is not None and len(choices) == 4
        opts = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(choices)])
        prompt = (
            f"{header} {question}\n"
            f"{opts}\n"
            f"{SYSTEM_STYLE_NOTE}\n"
            "Answer with exactly one letter (A, B, C, or D) inside <answer></answer>."
        )
        return prompt

    raise ValueError(f"Unknown task: {task}")

def gold_completion(gold_text: str) -> str:
    return f"<answer>{gold_text}</answer>"

# ============================================================
# DATA / ANSWER HELPERS
# ============================================================

def canonical_number_str(x):
    if x is None:
        return None
    try:
        v = float(re.sub(r"[$,]", "", str(x)))
    except Exception:
        return None
    if abs(v - round(v)) < 1e-6:
        return str(int(round(v)))
    return f"{v:.6f}".rstrip("0").rstrip(".")

def same_numeric(a, b, tol=1e-6):
    try:
        na = float(re.sub(r"[$,]", "", str(a)))
        nb = float(re.sub(r"[$,]", "", str(b)))
        return abs(na - nb) <= tol
    except Exception:
        return False

def extract_number(text):
    # Prefer anything in <answer>...</answer>, then fall back to the last number.
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = m[-1] if m else text
    span = re.sub(r"[$,]", "", span)
    nums = re.findall(r"-?\d+\.?\d*", span)
    if nums:
        return nums[-1]
    nums = re.findall(r"-?\d+\.?\d*", re.sub(r"[$,]", "", text))
    return nums[-1] if nums else None

def extract_yes_no(text):
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = m[-1] if m else text
    span = span.lower()
    hits = re.findall(r"\b(yes|no)\b", span, flags=re.IGNORECASE)
    if hits:
        return hits[-1].lower()
    # fallback to the last yes/no in the whole output
    hits = re.findall(r"\b(yes|no)\b", text, flags=re.IGNORECASE)
    return hits[-1].lower() if hits else None

def extract_mcq(text):
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = (m[-1] if m else text).upper()
    if span.strip() in ["A", "B", "C", "D"]:
        return span.strip()

    # common answer phrasings
    patterns = [
        r"ANSWER\s*[:\-]?\s*\(?([ABCD])\)?",
        r"\b([ABCD])\b\s*$",
        r"<ANSWER>\s*\(?([ABCD])\)?",
    ]
    for pat in patterns:
        matches = re.findall(pat, span)
        if matches:
            return matches[-1].upper()

    letters = re.findall(r"\b([ABCD])\b", span)
    if letters:
        return letters[-1].upper()

    return None

def parse_prediction(task, text):
    if task == "gsm8k":
        return canonical_number_str(extract_number(text))
    if task == "strategyqa":
        return extract_yes_no(text)
    if task == "mmlu":
        return extract_mcq(text)
    raise ValueError(task)

def exact_correct(task, pred, gold):
    if task == "gsm8k":
        return same_numeric(pred, gold)
    return pred == gold

def preview_text(s, max_len=220):
    s = str(s).replace("\n", " ")
    return s[:max_len] + ("..." if len(s) > max_len else "")

def to_jsonable(x):
    try:
        return json.dumps(x, ensure_ascii=False)
    except Exception:
        return json.dumps(str(x), ensure_ascii=False)

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def free_memory(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.synchronize()
        except Exception:
            pass

# ============================================================
# MODEL LOADING
# ============================================================

def load_tokenizer():
    global TOKENIZER
    if TOKENIZER is None:
        src = SFT_PATH if (USE_SFT_ADAPTER and os.path.exists(SFT_PATH)) else BASE_MODEL
        TOKENIZER = AutoTokenizer.from_pretrained(src)
        if TOKENIZER.pad_token is None:
            TOKENIZER.pad_token = TOKENIZER.eos_token
    return TOKENIZER

def load_base_model():
    kwargs = dict(torch_dtype=DTYPE)
    try:
        kwargs["attn_implementation"] = "eager"
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)
    except Exception:
        kwargs.pop("attn_implementation", None)
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)

    model = model.to(DEVICE)

    if USE_SFT_ADAPTER and os.path.exists(SFT_PATH) and HAS_PEFT:
        peft_model = PeftModel.from_pretrained(model, SFT_PATH)
        model = peft_model.merge_and_unload()
        model = model.to(DEVICE)

    model.eval()
    try:
        model.config.use_cache = False
    except Exception:
        pass
    return model

def discover_blocks(model):
    candidates = [
        ("model.layers", getattr(getattr(model, "model", None), "layers", None)),
        ("model.decoder.layers", getattr(getattr(getattr(model, "model", None), "decoder", None), "layers", None)),
        ("transformer.h", getattr(getattr(model, "transformer", None), "h", None)),
        ("gpt_neox.layers", getattr(getattr(model, "gpt_neox", None), "layers", None)),
        ("decoder.layers", getattr(getattr(model, "decoder", None), "layers", None)),
    ]
    for name, blocks in candidates:
        if blocks is not None:
            blocks = list(blocks)
            if len(blocks) > 0:
                return blocks, name
    raise RuntimeError("Could not locate transformer blocks. Try inspecting model architecture manually.")

def discover_final_norm(model):
    paths = [
        ("model.norm", getattr(getattr(model, "model", None), "norm", None)),
        ("transformer.ln_f", getattr(getattr(model, "transformer", None), "ln_f", None)),
        ("gpt_neox.final_layer_norm", getattr(getattr(model, "gpt_neox", None), "final_layer_norm", None)),
        ("decoder.final_layer_norm", getattr(getattr(model, "decoder", None), "final_layer_norm", None)),
    ]
    for name, mod in paths:
        if mod is not None:
            return mod, name
    return None, None

def discover_lm_head(model):
    if hasattr(model, "lm_head"):
        return model.lm_head, "lm_head"
    if hasattr(model, "embed_out"):
        return model.embed_out, "embed_out"
    raise RuntimeError("Could not locate lm_head / embed_out.")

def initialize_model():
    global MODEL, BLOCKS, FINAL_NORM, LM_HEAD, TOKENIZER
    TOKENIZER = load_tokenizer()
    MODEL = load_base_model()
    BLOCKS, block_path = discover_blocks(MODEL)
    FINAL_NORM, norm_path = discover_final_norm(MODEL)
    LM_HEAD, lm_path = discover_lm_head(MODEL)
    print(f"Model loaded. Blocks: {len(BLOCKS)} ({block_path}), norm: {norm_path}, head: {lm_path}")

# ============================================================
# TOKENIZATION / PROMPTS
# ============================================================

def tokenize_text(tokenizer, text):
    return tokenizer(text, return_tensors="pt", add_special_tokens=True).input_ids[0].tolist()

def token_preview(tokenizer, text, n=12):
    ids = tokenize_text(tokenizer, text)
    toks = tokenizer.convert_ids_to_tokens(ids[:n])
    return {
        "n_tokens": len(ids),
        "first_tokens": to_jsonable(toks),
        "first_ids": to_jsonable(ids[:n]),
    }

# ============================================================
# DATASET SAMPLING
# ============================================================

def cache_sample_indices(name, ds_len, n, seed=EVAL_SEED):
    ensure_dir(CACHE_DIR)
    n = min(n, ds_len)
    path = os.path.join(CACHE_DIR, f"{name}_n{n}_seed{seed}.json")
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    rng = np.random.RandomState(seed)
    idx = rng.choice(ds_len, size=n, replace=False).tolist()
    with open(path, "w") as f:
        json.dump(idx, f)
    return idx

def sample_eval_dataset(ds, name, n, seed=EVAL_SEED):
    idx = cache_sample_indices(name, len(ds), n, seed=seed)
    return ds.select(idx), idx

def load_task_samples(task):
    if task == "gsm8k":
        ds = load_dataset("gsm8k", "main", split="test")
        ds, idx = sample_eval_dataset(ds, "gsm8k", N_GSM8K)
        samples = []
        for i, s in enumerate(ds):
            gold = canonical_number_str(s["answer"].split("####")[-1].strip())
            samples.append({
                "sample_id": i,
                "task": task,
                "question": s["question"],
                "choices": None,
                "gold": gold,
                "source_index": idx[i],
            })
        return samples

    if task == "strategyqa":
        ds = load_dataset("ChilleD/StrategyQA", split="test")
        ds, idx = sample_eval_dataset(ds, "strategyqa", N_STRATEGYQA)
        samples = []
        for i, s in enumerate(ds):
            gold = "yes" if bool(s["answer"]) else "no"
            samples.append({
                "sample_id": i,
                "task": task,
                "question": s["question"],
                "choices": None,
                "gold": gold,
                "source_index": idx[i],
            })
        return samples

    if task == "mmlu":
        all_samples = []
        for sub in MMLU_SUBJECTS:
            ds = load_dataset("cais/mmlu", sub, split="test")
            ds, idx = sample_eval_dataset(ds, f"mmlu_{sub}", N_MMLU_PER_SUBJECT)
            for i, s in enumerate(ds):
                gold = chr(65 + int(s["answer"]))
                all_samples.append({
                    "sample_id": len(all_samples),
                    "task": task,
                    "subject": sub,
                    "question": s["question"],
                    "choices": list(s["choices"]),
                    "gold": gold,
                    "source_index": idx[i],
                })
        return all_samples

    raise ValueError(f"Unknown task: {task}")

# ============================================================
# GENERATION / SCORING
# ============================================================

def generate_greedy(prompt, max_new_tokens):
    tokenizer = TOKENIZER
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out = MODEL.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text

def sequence_logprob(prompt, completion, max_length=4096):
    tokenizer = TOKENIZER
    full_text = prompt + completion
    prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length).input_ids.to(DEVICE)
    full_ids = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=max_length).input_ids.to(DEVICE)

    with torch.inference_mode():
        logits = MODEL(full_ids).logits[:, :-1, :]
        logp_all = torch.log_softmax(logits.float(), dim=-1)
        tgt = full_ids[:, 1:]
        tok_lp = logp_all.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)

    start = max(0, prompt_ids.shape[1] - 1)
    comp_lp = tok_lp[:, start:]
    if comp_lp.numel() == 0:
        return {
            "sum_logprob": float("-inf"),
            "mean_logprob": float("-inf"),
            "n_tokens": 0,
        }

    return {
        "sum_logprob": float(comp_lp.sum().item()),
        "mean_logprob": float(comp_lp.mean().item()),
        "n_tokens": int(comp_lp.numel()),
    }

def first_answer_token_id(answer_text):
    tokenizer = TOKENIZER
    # Prefer a leading-space token because generation after the prompt usually begins with space/word boundary.
    candidates = [
        tokenizer.encode(" " + str(answer_text), add_special_tokens=False),
        tokenizer.encode(str(answer_text), add_special_tokens=False),
    ]
    for ids in candidates:
        if len(ids) > 0:
            return int(ids[0])
    return int(tokenizer.eos_token_id)

def apply_final_norm(h):
    if FINAL_NORM is None:
        return h
    try:
        return FINAL_NORM(h)
    except Exception:
        return h

def target_logit_from_hidden(h, target_id):
    """
    DLA proxy: project a hidden state through the final norm + unembedding.
    """
    if h.ndim == 3:
        h = h[:, -1, :]
    elif h.ndim == 2:
        pass
    else:
        raise ValueError("Unexpected hidden state shape.")

    h = apply_final_norm(h)
    w = LM_HEAD.weight[target_id]
    return torch.dot(h.squeeze(0).float(), w.float()).item()

def forward_with_hidden_and_attn(prompt):
    tokenizer = TOKENIZER
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out = MODEL(
            **inputs,
            output_hidden_states=True,
            output_attentions=True,
            use_cache=False,
            return_dict=True,
        )
    return inputs, out

# ============================================================
# PARSING / ACCURACY
# ============================================================

def evaluate_generation(task, generated_text, gold):
    pred = parse_prediction(task, generated_text)
    return pred, exact_correct(task, pred, gold)

def answer_for_task(task, gold):
    return gold

# ============================================================
# ACTIVATION PATCHING
# ============================================================

def patch_hook_factory(clean_state, positions):
    """
    Replaces selected token positions in a block output with clean activations.
    clean_state: tensor [1, seq, hidden] after the target layer on the clean prompt.
    """
    def hook(module, inp, output):
        if isinstance(output, tuple):
            hidden = output[0]
            patched = hidden.clone()
            pos = [p for p in positions if p < hidden.shape[1] and p < clean_state.shape[1]]
            if pos:
                patched[:, pos, :] = clean_state[:, pos, :].to(device=patched.device, dtype=patched.dtype)
            return (patched,) + output[1:]
        else:
            patched = output.clone()
            pos = [p for p in positions if p < patched.shape[1] and p < clean_state.shape[1]]
            if pos:
                patched[:, pos, :] = clean_state[:, pos, :].to(device=patched.device, dtype=patched.dtype)
            return patched
    return hook

def patched_sequence_score(corrupt_prompt, completion, clean_hidden_state_after_layer, layer_idx, patch_positions):
    """
    Run the corrupted prompt with a forward hook on one layer that swaps in clean activations
    at the chosen positions. Score is mean logprob of the gold completion.
    """
    handle = BLOCKS[layer_idx].register_forward_hook(
        patch_hook_factory(clean_hidden_state_after_layer, patch_positions)
    )
    try:
        score = sequence_logprob(corrupt_prompt, completion)
    finally:
        handle.remove()
    return score

def choose_position_sweep_layers(num_layers, n=POSITION_SWEEP_LAYERS):
    if num_layers <= 1:
        return [0]
    grid = np.linspace(0, num_layers - 1, num=min(n, num_layers))
    return sorted(set(int(round(x)) for x in grid))

# ============================================================
# ATTENTION ANALYSIS
# ============================================================

def attention_mass(attentions, query_pos, prefix_positions):
    """
    attentions: tuple[layer] -> [1, heads, seq, seq]
    Returns:
      layer_mean: [n_layers]
      layer_head:  [n_layers, n_heads]
    """
    layer_mean = []
    layer_head = []
    for layer_attn in attentions:
        a = layer_attn[0]  # [heads, seq, seq]
        if query_pos >= a.shape[-2]:
            layer_mean.append(float("nan"))
            layer_head.append(np.full((a.shape[0],), np.nan))
            continue

        prefix_positions = [p for p in prefix_positions if p < a.shape[-1]]
        if not prefix_positions:
            layer_mean.append(float("nan"))
            layer_head.append(np.full((a.shape[0],), np.nan))
            continue

        # attention from final prompt token to prefix tokens
        mass_per_head = a[:, query_pos, prefix_positions].sum(dim=-1)  # [heads]
        layer_mean.append(float(mass_per_head.mean().item()))
        layer_head.append(mass_per_head.detach().float().cpu().numpy())

    return np.array(layer_mean), np.stack(layer_head, axis=0)

# ============================================================
# PLOTTING HELPERS
# ============================================================

def save_line_plot(x, ys, labels, title, path, xlabel="Layer", ylabel="Value"):
    plt.figure(figsize=(10, 5))
    for y, lab in zip(ys, labels):
        plt.plot(x, y, marker="o", linewidth=1.5, label=lab)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def save_heatmap(mat, xlabels, ylabels, title, path, xlabel="", ylabel="", cmap="viridis"):
    plt.figure(figsize=(max(8, len(xlabels) * 0.4), max(5, len(ylabels) * 0.35)))
    im = plt.imshow(mat, aspect="auto", cmap=cmap)
    plt.colorbar(im)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(range(len(xlabels)), xlabels, rotation=45, ha="right")
    plt.yticks(range(len(ylabels)), ylabels)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def save_bar_plot(labels, values, title, path, ylabel="Value"):
    plt.figure(figsize=(8, 4))
    plt.bar(labels, values)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

# ============================================================
# EXPERIMENT PER SAMPLE
# ============================================================

def run_sample(task, sample):
    q = sample["question"]
    choices = sample.get("choices", None)
    gold = sample["gold"]

    clean_prompt = build_prompt(task, q, clean=True, choices=choices)
    corrupt_prompt = build_prompt(task, q, clean=False, choices=choices)
    completion = gold_completion(answer_for_task(task, gold))

    # generation
    max_new = {
        "gsm8k": MAX_NEW_GSM8K,
        "strategyqa": MAX_NEW_STRATEGYQA,
        "mmlu": MAX_NEW_MMLU,
    }[task]

    clean_gen = generate_greedy(clean_prompt, max_new)
    corrupt_gen = generate_greedy(corrupt_prompt, max_new)

    clean_pred, clean_ok = evaluate_generation(task, clean_gen, gold)
    corrupt_pred, corrupt_ok = evaluate_generation(task, corrupt_gen, gold)

    # teacher-forced gold continuation logprob
    clean_score = sequence_logprob(clean_prompt, completion)
    corrupt_score = sequence_logprob(corrupt_prompt, completion)

    # hidden states / attentions for causal analysis
    clean_inputs, clean_out = forward_with_hidden_and_attn(clean_prompt)
    corrupt_inputs, corrupt_out = forward_with_hidden_and_attn(corrupt_prompt)

    clean_hidden = clean_out.hidden_states
    corrupt_hidden = corrupt_out.hidden_states
    clean_attn = clean_out.attentions
    corrupt_attn = corrupt_out.attentions

    seq_len_clean = int(clean_inputs["input_ids"].shape[1])
    seq_len_corrupt = int(corrupt_inputs["input_ids"].shape[1])
    qpos_clean = seq_len_clean - 1
    qpos_corrupt = seq_len_corrupt - 1

    # target token for DLA: first token of the gold answer content
    target_id = first_answer_token_id(gold)

    # DLA proxy curves: layer 0 = embeddings, layers 1..L = after each transformer block
    clean_dla = []
    corrupt_dla = []
    for layer_idx in range(len(clean_hidden)):
        clean_dla.append(target_logit_from_hidden(clean_hidden[layer_idx][:, qpos_clean:qpos_clean+1, :], target_id))
        corrupt_dla.append(target_logit_from_hidden(corrupt_hidden[layer_idx][:, qpos_corrupt:qpos_corrupt+1, :], target_id))

    # Activation patching across layers: patch the first PATCH_WINDOW positions
    patch_positions = list(range(min(PATCH_WINDOW, seq_len_clean, seq_len_corrupt)))
    patch_rows = []
    for layer_idx in range(len(BLOCKS)):
        # clean state after this layer = hidden_states[layer_idx + 1]
        clean_after = clean_hidden[layer_idx + 1]
        patched = patched_sequence_score(
            corrupt_prompt,
            completion,
            clean_after,
            layer_idx,
            patch_positions,
        )
        patch_rows.append({
            "sample_id": sample["sample_id"],
            "task": task,
            "layer": layer_idx,
            "clean_mean_logprob": clean_score["mean_logprob"],
            "corrupt_mean_logprob": corrupt_score["mean_logprob"],
            "patched_mean_logprob": patched["mean_logprob"],
            "clean_sum_logprob": clean_score["sum_logprob"],
            "corrupt_sum_logprob": corrupt_score["sum_logprob"],
            "patched_sum_logprob": patched["sum_logprob"],
            "recovery_mean": (
                (patched["mean_logprob"] - corrupt_score["mean_logprob"])
                / (clean_score["mean_logprob"] - corrupt_score["mean_logprob"] + 1e-8)
            ),
            "recovery_sum": (
                (patched["sum_logprob"] - corrupt_score["sum_logprob"])
                / (clean_score["sum_logprob"] - corrupt_score["sum_logprob"] + 1e-8)
            ),
            "patch_window": len(patch_positions),
        })

    # Position sweep on a few representative layers
    sweep_layers = choose_position_sweep_layers(len(BLOCKS), POSITION_SWEEP_LAYERS)
    pos_rows = []
    for layer_idx in sweep_layers:
        clean_after = clean_hidden[layer_idx + 1]
        for pos in range(min(PATCH_WINDOW, seq_len_clean, seq_len_corrupt)):
            patched = patched_sequence_score(
                corrupt_prompt,
                completion,
                clean_after,
                layer_idx,
                [pos],
            )
            pos_rows.append({
                "sample_id": sample["sample_id"],
                "task": task,
                "layer": layer_idx,
                "position": pos,
                "clean_mean_logprob": clean_score["mean_logprob"],
                "corrupt_mean_logprob": corrupt_score["mean_logprob"],
                "patched_mean_logprob": patched["mean_logprob"],
                "recovery_mean": (
                    (patched["mean_logprob"] - corrupt_score["mean_logprob"])
                    / (clean_score["mean_logprob"] - corrupt_score["mean_logprob"] + 1e-8)
                ),
            })

    # Attention mass to the first prefix tokens
    attn_rows = []
    if clean_attn is not None and corrupt_attn is not None:
        prefix_positions_clean = list(range(min(ATTN_PREFIX_WINDOW, seq_len_clean)))
        prefix_positions_corrupt = list(range(min(ATTN_PREFIX_WINDOW, seq_len_corrupt)))

        clean_layer_mean, clean_layer_head = attention_mass(clean_attn, qpos_clean, prefix_positions_clean)
        corrupt_layer_mean, corrupt_layer_head = attention_mass(corrupt_attn, qpos_corrupt, prefix_positions_corrupt)

        for layer_idx in range(len(clean_layer_mean)):
            attn_rows.append({
                "sample_id": sample["sample_id"],
                "task": task,
                "layer": layer_idx,
                "clean_prefix_mass_mean": float(clean_layer_mean[layer_idx]),
                "corrupt_prefix_mass_mean": float(corrupt_layer_mean[layer_idx]),
                "delta_prefix_mass_mean": float(clean_layer_mean[layer_idx] - corrupt_layer_mean[layer_idx]),
            })

        # head-level rows
        for layer_idx in range(clean_layer_head.shape[0]):
            for head_idx in range(clean_layer_head.shape[1]):
                attn_rows.append({
                    "sample_id": sample["sample_id"],
                    "task": task,
                    "layer": layer_idx,
                    "head": head_idx,
                    "clean_prefix_mass_head": float(clean_layer_head[layer_idx, head_idx]),
                    "corrupt_prefix_mass_head": float(corrupt_layer_head[layer_idx, head_idx]),
                    "delta_prefix_mass_head": float(clean_layer_head[layer_idx, head_idx] - corrupt_layer_head[layer_idx, head_idx]),
                    "is_head_row": True,
                })

    # tokenization diagnostics
    clean_tok = token_preview(TOKENIZER, clean_prompt, n=16)
    corrupt_tok = token_preview(TOKENIZER, corrupt_prompt, n=16)

    sample_row = {
        "sample_id": sample["sample_id"],
        "task": task,
        "subject": sample.get("subject", ""),
        "source_index": sample["source_index"],
        "question": q,
        "choices": to_jsonable(choices),
        "gold": gold,
        "clean_prompt_preview": preview_text(clean_prompt, 300),
        "corrupt_prompt_preview": preview_text(corrupt_prompt, 300),
        "clean_prediction": clean_pred,
        "corrupt_prediction": corrupt_pred,
        "clean_correct": bool(clean_ok),
        "corrupt_correct": bool(corrupt_ok),
        "clean_raw_output": preview_text(clean_gen, 500),
        "corrupt_raw_output": preview_text(corrupt_gen, 500),
        "clean_gold_mean_logprob": clean_score["mean_logprob"],
        "corrupt_gold_mean_logprob": corrupt_score["mean_logprob"],
        "clean_gold_sum_logprob": clean_score["sum_logprob"],
        "corrupt_gold_sum_logprob": corrupt_score["sum_logprob"],
        "logprob_delta_mean": clean_score["mean_logprob"] - corrupt_score["mean_logprob"],
        "logprob_delta_sum": clean_score["sum_logprob"] - corrupt_score["sum_logprob"],
        "clean_n_tokens": clean_tok["n_tokens"],
        "corrupt_n_tokens": corrupt_tok["n_tokens"],
        "clean_first_tokens": clean_tok["first_tokens"],
        "corrupt_first_tokens": corrupt_tok["first_tokens"],
        "clean_first_ids": clean_tok["first_ids"],
        "corrupt_first_ids": corrupt_tok["first_ids"],
        "target_token_id": target_id,
        "target_token_str": TOKENIZER.convert_ids_to_tokens([target_id])[0] if target_id is not None else "",
        "prefix_patch_window": len(patch_positions),
    }

    dla_rows = []
    for layer_idx in range(len(clean_dla)):
        dla_rows.append({
            "sample_id": sample["sample_id"],
            "task": task,
            "layer": layer_idx,
            "clean_dla": clean_dla[layer_idx],
            "corrupt_dla": corrupt_dla[layer_idx],
            "delta_dla": clean_dla[layer_idx] - corrupt_dla[layer_idx],
        })

    return sample_row, patch_rows, pos_rows, dla_rows, attn_rows

# ============================================================
# AGGREGATION / PLOTTING
# ============================================================

def aggregate_and_plot(task, sample_rows, patch_rows, pos_rows, dla_rows, attn_rows):
    task_out = TASK_DIRS[task]
    ensure_dir(task_out)

    df_samples = pd.DataFrame(sample_rows)
    df_patch = pd.DataFrame(patch_rows)
    df_pos = pd.DataFrame(pos_rows)
    df_dla = pd.DataFrame(dla_rows)
    df_attn = pd.DataFrame(attn_rows)

    df_samples.to_csv(os.path.join(task_out, f"{task}_samples.csv"), index=False)
    df_patch.to_csv(os.path.join(task_out, f"{task}_activation_patch_layer.csv"), index=False)
    df_pos.to_csv(os.path.join(task_out, f"{task}_activation_patch_position.csv"), index=False)
    df_dla.to_csv(os.path.join(task_out, f"{task}_dla.csv"), index=False)
    df_attn.to_csv(os.path.join(task_out, f"{task}_attention.csv"), index=False)

    # tokenization CSV only (explicit subset)
    tok_cols = [
        "sample_id", "task", "source_index", "clean_n_tokens", "corrupt_n_tokens",
        "clean_first_tokens", "corrupt_first_tokens", "clean_first_ids", "corrupt_first_ids",
    ]
    df_tokens = df_samples[tok_cols].copy()
    df_tokens.to_csv(os.path.join(task_out, f"{task}_tokenization.csv"), index=False)

    # summary metrics
    summary = {
        "task": task,
        "n_samples": int(len(df_samples)),
        "clean_accuracy": float(df_samples["clean_correct"].mean()),
        "corrupt_accuracy": float(df_samples["corrupt_correct"].mean()),
        "clean_mean_gold_logprob": float(df_samples["clean_gold_mean_logprob"].mean()),
        "corrupt_mean_gold_logprob": float(df_samples["corrupt_gold_mean_logprob"].mean()),
        "mean_logprob_delta": float(df_samples["logprob_delta_mean"].mean()),
        "mean_clean_tokens": float(df_samples["clean_n_tokens"].mean()),
        "mean_corrupt_tokens": float(df_samples["corrupt_n_tokens"].mean()),
    }

    # patching aggregate
    patch_agg = df_patch.groupby("layer", as_index=False).agg(
        clean_mean_logprob=("clean_mean_logprob", "mean"),
        corrupt_mean_logprob=("corrupt_mean_logprob", "mean"),
        patched_mean_logprob=("patched_mean_logprob", "mean"),
        recovery_mean=("recovery_mean", "mean"),
        clean_sum_logprob=("clean_sum_logprob", "mean"),
        corrupt_sum_logprob=("corrupt_sum_logprob", "mean"),
        patched_sum_logprob=("patched_sum_logprob", "mean"),
        recovery_sum=("recovery_sum", "mean"),
    )

    pos_agg = df_pos.groupby(["layer", "position"], as_index=False).agg(
        patched_mean_logprob=("patched_mean_logprob", "mean"),
        recovery_mean=("recovery_mean", "mean"),
    )

    dla_agg = df_dla.groupby("layer", as_index=False).agg(
        clean_dla=("clean_dla", "mean"),
        corrupt_dla=("corrupt_dla", "mean"),
        delta_dla=("delta_dla", "mean"),
    )

    if len(df_attn) > 0 and "head" not in df_attn.columns:
        attn_layer_agg = df_attn.groupby("layer", as_index=False).agg(
            clean_prefix_mass_mean=("clean_prefix_mass_mean", "mean"),
            corrupt_prefix_mass_mean=("corrupt_prefix_mass_mean", "mean"),
            delta_prefix_mass_mean=("delta_prefix_mass_mean", "mean"),
        )
    else:
        attn_layer_agg = pd.DataFrame()

    # write aggregate CSVs
    patch_agg.to_csv(os.path.join(task_out, f"{task}_activation_patch_layer_agg.csv"), index=False)
    pos_agg.to_csv(os.path.join(task_out, f"{task}_activation_patch_position_agg.csv"), index=False)
    dla_agg.to_csv(os.path.join(task_out, f"{task}_dla_agg.csv"), index=False)
    if len(attn_layer_agg) > 0:
        attn_layer_agg.to_csv(os.path.join(task_out, f"{task}_attention_layer_agg.csv"), index=False)

    # plots
    layers = patch_agg["layer"].tolist()
    save_line_plot(
        layers,
        [
            patch_agg["clean_mean_logprob"].tolist(),
            patch_agg["corrupt_mean_logprob"].tolist(),
            patch_agg["patched_mean_logprob"].tolist(),
        ],
        ["clean", "corrupt", "patched"],
        f"{task.upper()} gold continuation score by layer",
        os.path.join(PLOTS_DIR, f"{task}_patch_recovery_curve.png"),
        xlabel="Patched layer",
        ylabel="Mean gold continuation logprob",
    )

    save_line_plot(
        layers,
        [
            patch_agg["recovery_mean"].tolist(),
        ],
        ["normalized recovery"],
        f"{task.upper()} normalized recovery by layer",
        os.path.join(PLOTS_DIR, f"{task}_patch_normalized_recovery.png"),
        xlabel="Patched layer",
        ylabel="Recovery",
    )

    save_line_plot(
        dla_agg["layer"].tolist(),
        [
            dla_agg["clean_dla"].tolist(),
            dla_agg["corrupt_dla"].tolist(),
            dla_agg["delta_dla"].tolist(),
        ],
        ["clean", "corrupt", "delta"],
        f"{task.upper()} DLA proxy by layer",
        os.path.join(PLOTS_DIR, f"{task}_dla_curve.png"),
        xlabel="Layer",
        ylabel="DLA proxy",
    )

    if len(attn_layer_agg) > 0:
        save_line_plot(
            attn_layer_agg["layer"].tolist(),
            [
                attn_layer_agg["clean_prefix_mass_mean"].tolist(),
                attn_layer_agg["corrupt_prefix_mass_mean"].tolist(),
                attn_layer_agg["delta_prefix_mass_mean"].tolist(),
            ],
            ["clean", "corrupt", "delta"],
            f"{task.upper()} prefix attention mass by layer",
            os.path.join(PLOTS_DIR, f"{task}_attention_curve.png"),
            xlabel="Layer",
            ylabel=f"Mean attention mass to first {ATTN_PREFIX_WINDOW} tokens",
        )

    # patch position heatmap
    if len(pos_agg) > 0:
        layers_sorted = sorted(pos_agg["layer"].unique().tolist())
        pos_sorted = sorted(pos_agg["position"].unique().tolist())
        heat = np.full((len(layers_sorted), len(pos_sorted)), np.nan)
        for _, r in pos_agg.iterrows():
            i = layers_sorted.index(int(r["layer"]))
            j = pos_sorted.index(int(r["position"]))
            heat[i, j] = r["recovery_mean"]
        save_heatmap(
            heat,
            [str(p) for p in pos_sorted],
            [str(l) for l in layers_sorted],
            f"{task.upper()} patch recovery heatmap",
            os.path.join(PLOTS_DIR, f"{task}_patch_position_heatmap.png"),
            xlabel="Patched position",
            ylabel="Layer",
            cmap="viridis",
        )

    # attention head heatmap
    if len(df_attn) > 0 and "head" in df_attn.columns:
        head_agg = df_attn.groupby(["layer", "head"], as_index=False).agg(
            delta_prefix_mass_head=("delta_prefix_mass_head", "mean"),
            clean_prefix_mass_head=("clean_prefix_mass_head", "mean"),
            corrupt_prefix_mass_head=("corrupt_prefix_mass_head", "mean"),
        )
        head_layers = sorted(head_agg["layer"].unique().tolist())
        head_ids = sorted(head_agg["head"].unique().tolist())
        heat = np.full((len(head_layers), len(head_ids)), np.nan)
        for _, r in head_agg.iterrows():
            i = head_layers.index(int(r["layer"]))
            j = head_ids.index(int(r["head"]))
            heat[i, j] = r["delta_prefix_mass_head"]
        save_heatmap(
            heat,
            [str(h) for h in head_ids],
            [str(l) for l in head_layers],
            f"{task.upper()} attention head delta heatmap",
            os.path.join(PLOTS_DIR, f"{task}_attention_head_heatmap.png"),
            xlabel="Head",
            ylabel="Layer",
            cmap="coolwarm",
        )
        head_agg.to_csv(os.path.join(task_out, f"{task}_attention_head_agg.csv"), index=False)

        # top heads table
        top_heads = head_agg.sort_values("delta_prefix_mass_head", ascending=False).head(12)
        top_heads.to_csv(os.path.join(task_out, f"{task}_top_attention_heads.csv"), index=False)

    # tokenization histograms
    plt.figure(figsize=(8, 4))
    plt.hist(df_samples["clean_n_tokens"], bins=10, alpha=0.7, label="clean")
    plt.hist(df_samples["corrupt_n_tokens"], bins=10, alpha=0.7, label="corrupt")
    plt.title(f"{task.upper()} prompt token length distribution")
    plt.xlabel("Tokens")
    plt.ylabel("Count")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, f"{task}_token_length_hist.png"), dpi=180)
    plt.close()

    # accuracy bar plot
    save_bar_plot(
        ["clean", "corrupt"],
        [summary["clean_accuracy"], summary["corrupt_accuracy"]],
        f"{task.upper()} generation accuracy",
        os.path.join(PLOTS_DIR, f"{task}_accuracy_bar.png"),
        ylabel="Accuracy",
    )

    # logprob bar plot
    save_bar_plot(
        ["clean", "corrupt"],
        [summary["clean_mean_gold_logprob"], summary["corrupt_mean_gold_logprob"]],
        f"{task.upper()} gold continuation score",
        os.path.join(PLOTS_DIR, f"{task}_gold_logprob_bar.png"),
        ylabel="Mean gold continuation logprob",
    )

    # save summary
    with open(os.path.join(task_out, f"{task}_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    # markdown report
    md = []
    md.append(f"# {task.upper()} delimiter causality report\n")
    md.append("| Metric | Clean | Corrupt | Delta |\n|---|---:|---:|---:|")
    md.append(f"| Exact accuracy | {summary['clean_accuracy']:.3f} | {summary['corrupt_accuracy']:.3f} | {summary['clean_accuracy'] - summary['corrupt_accuracy']:+.3f} |")
    md.append(f"| Mean gold logprob | {summary['clean_mean_gold_logprob']:.3f} | {summary['corrupt_mean_gold_logprob']:.3f} | {summary['mean_logprob_delta']:+.3f} |")
    md.append(f"| Mean prompt tokens | {summary['mean_clean_tokens']:.1f} | {summary['mean_corrupt_tokens']:.1f} | {summary['mean_clean_tokens'] - summary['mean_corrupt_tokens']:+.1f} |")
    md.append("\n## Key observations\n")
    best_layer = int(patch_agg.sort_values("recovery_mean", ascending=False).iloc[0]["layer"])
    best_recovery = float(patch_agg.sort_values("recovery_mean", ascending=False).iloc[0]["recovery_mean"])
    best_dla_layer = int(dla_agg.sort_values("delta_dla", ascending=False).iloc[0]["layer"])
    best_dla_delta = float(dla_agg.sort_values("delta_dla", ascending=False).iloc[0]["delta_dla"])
    md.append(f"- Strongest patch recovery layer: **{best_layer}** (normalized recovery {best_recovery:.3f}).")
    md.append(f"- Strongest DLA delta layer: **{best_dla_layer}** (delta {best_dla_delta:.3f}).")
    if len(attn_layer_agg) > 0:
        best_attn_layer = int(attn_layer_agg.sort_values("delta_prefix_mass_mean", ascending=False).iloc[0]["layer"])
        best_attn_delta = float(attn_layer_agg.sort_values("delta_prefix_mass_mean", ascending=False).iloc[0]["delta_prefix_mass_mean"])
        md.append(f"- Strongest attention-prefix delta layer: **{best_attn_layer}** (delta {best_attn_delta:.3f}).")
    md_text = "\n".join(md)
    with open(os.path.join(task_out, f"{task}_report.md"), "w") as f:
        f.write(md_text)

    return {
        "summary": summary,
        "patch_agg": patch_agg,
        "pos_agg": pos_agg,
        "dla_agg": dla_agg,
        "attn_layer_agg": attn_layer_agg,
    }

# ============================================================
# MAIN
# ============================================================

def main():
    initialize_model()

    all_task_summaries = []
    combined = {
        "samples": [],
        "patch_rows": [],
        "pos_rows": [],
        "dla_rows": [],
        "attn_rows": [],
    }

    for task in ["gsm8k", "strategyqa", "mmlu"]:
        print(f"\n===== Running {task.upper()} =====")
        samples = load_task_samples(task)

        task_sample_rows = []
        task_patch_rows = []
        task_pos_rows = []
        task_dla_rows = []
        task_attn_rows = []

        for sample in samples:
            print(f"  sample {sample['sample_id']}/{len(samples)-1}: {preview_text(sample['question'], 80)}")
            srow, prows, ppos, drows, arows = run_sample(task, sample)
            task_sample_rows.append(srow)
            task_patch_rows.extend(prows)
            task_pos_rows.extend(ppos)
            task_dla_rows.extend(drows)
            task_attn_rows.extend(arows)

            # free transient GPU memory
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        result = aggregate_and_plot(task, task_sample_rows, task_patch_rows, task_pos_rows, task_dla_rows, task_attn_rows)
        all_task_summaries.append(result["summary"])

        combined["samples"].extend(task_sample_rows)
        combined["patch_rows"].extend(task_patch_rows)
        combined["pos_rows"].extend(task_pos_rows)
        combined["dla_rows"].extend(task_dla_rows)
        combined["attn_rows"].extend(task_attn_rows)

    # combined summary
    summary = {
        "tasks": all_task_summaries,
        "n_tasks": len(all_task_summaries),
        "config": {
            "BASE_MODEL": BASE_MODEL,
            "SFT_PATH": SFT_PATH,
            "USE_SFT_ADAPTER": USE_SFT_ADAPTER,
            "USE_FEWSHOT": USE_FEWSHOT,
            "PATCH_WINDOW": PATCH_WINDOW,
            "POSITION_SWEEP_LAYERS": POSITION_SWEEP_LAYERS,
            "ATTN_PREFIX_WINDOW": ATTN_PREFIX_WINDOW,
            "N_GSM8K": N_GSM8K,
            "N_STRATEGYQA": N_STRATEGYQA,
            "N_MMLU_PER_SUBJECT": N_MMLU_PER_SUBJECT,
            "MMLU_SUBJECTS": MMLU_SUBJECTS,
        }
    }

    with open(os.path.join(REPORTS_DIR, "summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    # global CSVs
    pd.DataFrame(combined["samples"]).to_csv(os.path.join(OUT_DIR, "all_samples.csv"), index=False)
    pd.DataFrame(combined["patch_rows"]).to_csv(os.path.join(OUT_DIR, "all_activation_patch_layer.csv"), index=False)
    pd.DataFrame(combined["pos_rows"]).to_csv(os.path.join(OUT_DIR, "all_activation_patch_position.csv"), index=False)
    pd.DataFrame(combined["dla_rows"]).to_csv(os.path.join(OUT_DIR, "all_dla.csv"), index=False)
    pd.DataFrame(combined["attn_rows"]).to_csv(os.path.join(OUT_DIR, "all_attention.csv"), index=False)

    # global markdown summary
    md = ["# Delimiter causality experiment summary\n"]
    for t in summary["tasks"]:
        md.append(f"## {t['task'].upper()}")
        md.append("| Metric | Value |\n|---|---:|")
        md.append(f"| Clean accuracy | {t['clean_accuracy']:.3f} |")
        md.append(f"| Corrupt accuracy | {t['corrupt_accuracy']:.3f} |")
        md.append(f"| Mean gold logprob delta | {t['mean_logprob_delta']:.3f} |")
        md.append("")
    with open(os.path.join(REPORTS_DIR, "summary.md"), "w") as f:
        f.write("\n".join(md))

    print("\nDone.")
    print(f"Outputs saved to: {OUT_DIR}")
    print(f"Plots: {PLOTS_DIR}")
    print(f"Reports: {REPORTS_DIR}")

if __name__ == "__main__":
    main()

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

...

logical_fallacies/dev-00000-of-00001.par(…):   0%|          | 0.00/4.12k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/163 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/18 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

  sample 0/11: Statement 1 | If a group has an element of order 10, then it has elements of ord...
  sample 1/11: Statement 1 | If G, H and K are groups of order 4, at least two of them are isom...
  sample 2/11: (Z,*) is a group with a*b = a+b+1 for all a, b in Z. The inverse of a is
  sample 3/11: Statement 1 | For any two groups G and G', there exists a homomorphism of G into...
  sample 4/11: If the finite group G contains a subgroup of order seven but no element (other t...
  sample 5/11: What is the area of an equilateral triangle whose inscribed circle has radius 2?
  sample 6/11: Water drips out of a hole at the vertex of an upside down cone at a rate of 3 cm...
  sample 7/11: Statement 1 | f : X → Y is continuous and X is compact. f must be uniformly cont...
  sample 8/11: Tan ah Tiat, forty-nine years old, a native of Kuala Lumpar, Malaysia, was charg...
  sample 9/11: Men are better drivers than women are. The proof of this is that men are more ca...
  sample 10/11: Which fa